# Results for model: mistralai/mistral-small-24b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

# Load data
data = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering
data = data.sort('date_id')
data = data.with_column(pl.col('feature_00').cast(pl.Float64))
data = data.with_column((pl.col('date_id').cumcount().over('date_id') / 100).cast(pl.Int32).alias('batch_id'))
data = data.with_column(pl.col('date_id').cast(pl.Datetime))

top_quantile = []
window_size = 100

for i in range(0, len(data), window_size):
    batch = data[i:i+window_size]
    top_15_percentile = batch.select(pl.quantile('feature_00', 0.85).alias('top_quantile')).to_series().to_list()[0]
    top_quantile.extend([top_15_percentile] * len(batch))

data = data.with_column(pl.Series("top_quantile", top_quantile))

# Prepare data for modeling
le = LabelEncoder()
data = data.with_column(le.fit_transform(data['batch_id']).alias('batch_id_int'))
features = data.select(['feature_00', 'batch_id_int', 'top_quantile'])
target = data['responder_6']

# Split data
X_train, X_test, y_train, y_test = train_test_split(features.to_pandas(), target.to_pandas(), test_size=0.2, random_state=42)

# Train XGBoost Regressor
model = xgb.XGBRFRegressor(objective='reg:squarederror', random_seed=42)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Save model
model.save_model('xgboost_model.json')